# Dynamic Workflow로 대규모 Subagent Orchestration하기

## 소개

투자자 업데이트 초안을 fact-check하려면 여러 독립적인 주장을 확인하고, 각각에 실제 판단을 적용하며, 검증이 실제로 실행되도록 해야 합니다. 하나의 context window에 편하게 담기에는 항목, 판단, 강제 검증이 너무 많습니다. Dynamic workflow는 하나의 대화가 조정할 수 있는 것보다 많은 agent가 필요한 이런 작업을 위한 Claude Code 기능입니다. Turn마다 orchestration하는 대신 Claude는 작업에 맞는 JavaScript orchestration script를 작성하여 `Workflow` tool에 전달하고, runtime은 이를 background에서 실행합니다. Script는 subagent를 병렬로 또는 단계별로 시작하고 결과를 변수에 보유하며, 연결 로직(filtering, loop, 검증 pass)을 일반 코드로 적용합니다. Agent SDK가 Claude Code runtime을 구동하므로 Python에서 workflow를 바로 시작할 수 있습니다.

각 subagent는 완전한 Claude Code agent입니다. 새로운 context로 시작하고 script가 제공한 prompt만 보며, 구성한 tool allowlist를 사용해 session의 작업 디렉터리에서 작업합니다. 기본적으로 모든 agent는 session의 model로 실행되지만 script가 단계를 다른 model로 보낼 수 있습니다. 기계적인 추출에는 저렴한 model, 어려운 판단에는 가장 강력한 model을 사용할 수 있습니다. Prompt에서 이를 안내하거나 `CLAUDE_CODE_SUBAGENT_MODEL` 환경 변수로 모든 agent를 한 번에 override합니다. Runtime은 최대 16개 agent를 동시에 실행하며(CPU core가 적은 컴퓨터에서는 더 적음), 실행당 1,000개 agent로 제한합니다. Concurrency 한도보다 많은 작업을 계획한 workflow는 slot이 비워질 때까지 queue에 넣습니다.

계획이 model의 context window 대신 코드에 존재하므로, 모든 항목을 동일하게 처리하고, script가 지시하므로 검증이 실행되며, 한 번의 실행이 수십에서 수백 agent를 조정할 수 있습니다. Script는 읽고, 수정하고, 저장하고, 다시 실행할 수 있는 파일입니다. 비용은 token입니다. 많은 subagent는 훨씬 많은 token을 사용합니다. 작업이 하나의 context window보다 크거나, 구조로 검증을 강제해야 하거나, orchestration 자체를 보존할 가치가 있을 때 workflow를 사용할 가치가 있습니다.

**이 cookbook을 마치면 다음을 할 수 있습니다:**

- 단일 agent나 subagent 대신 dynamic workflow가 필요한 작업 판단하기
- Python Agent SDK에서 dynamic workflow를 시작하고 진행 상황을 streaming하기
- Claude가 작성한 orchestration script를 읽고 내부 pattern(fan-out, adversarial verification) 알아보기

예제는 source 문서를 기준으로 투자자 업데이트 초안을 fact-check합니다. 많은 부분, 실제 판단, 검증 단계라는 동일한 형태는 문서 audit, 콘텐츠 migration, 조사 종합, triage queue에서도 나타납니다.

> **참고:** 이 Notebook의 모든 회사명, metrics, 고객 data는 시연을 위해 생성한 가상의 정보입니다.

## 사전 요구사항

**필수 지식:**

- `async`/`await`를 포함한 Python 기초
- Claude Agent SDK 기초(`query()` 및 `ClaudeAgentOptions`)에 익숙함. 처음이라면 [Notebook 00](00_The_one_liner_research_agent.ipynb)을 참고하세요.

**필수 Tool:**

- Python 3.11 이상
- `.env` 파일에 `ANTHROPIC_API_KEY`로 설정한 Anthropic API key([여기에서 발급](https://platform.claude.com/))

Agent SDK에는 구동 대상인 Claude Code CLI가 포함되며, dynamic workflow는 해당 runtime에 있습니다. CLI **v2.1.154 이상**이 필요하며, `claude-agent-sdk` 0.2.90 이상에 포함됩니다. 아래 설치 cell이 최소 version을 설정하므로 별도 설정은 없습니다.

Dynamic workflow는 모든 유료 plan, Anthropic API 접근, Amazon Bedrock, Google Cloud의 Agent Platform, Microsoft Foundry에서 사용할 수 있습니다. 현재 상세 내용은 [dynamic workflow 문서](https://code.claude.com/docs/en/workflows)를 참고하세요.

> **비용 참고:** Workflow는 많은 subagent를 시작하므로 단일 agent session보다 눈에 띄게 많은 token을 사용합니다. 이 Notebook 전체 실행에는 API 사용량으로 약 $2–$4가 들며, 단계 실패 후 재시도하면 더 듭니다. Workflow 실행은 완료 시 비용을 출력합니다.

## 설정

SDK를 설치하고 API key를 load합니다. 아래 최소 version은 포함된 Claude Code CLI가 dynamic workflow를 지원하는 SDK release를 보장하며, 다음 cell에서 확인합니다. 이미 이전 version을 import한 kernel에서 SDK를 upgrade했다면 Python이 새 version을 사용하도록 kernel을 재시작하세요.

In [1]:
%%capture
%pip install -U "claude-agent-sdk>=0.2.90" python-dotenv

In [2]:
import re
import shutil
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

import claude_agent_sdk
from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    TaskNotificationMessage,
    TaskProgressMessage,
    TextBlock,
    ToolResultBlock,
    ToolUseBlock,
    UserMessage,
    query,
)

load_dotenv()

MODEL = "claude-sonnet-5"

# Dynamic workflow에는 Claude Code v2.1.154+가 필요하며 SDK 0.2.90 이상에 호환 CLI가 포함됨.
version_match = re.match(r"(\d+)\.(\d+)\.(\d+)", claude_agent_sdk.__version__)
sdk_version = tuple(int(g) for g in version_match.groups()) if version_match else (0, 0, 0)
assert sdk_version >= (0, 2, 90), (
    f"claude-agent-sdk {claude_agent_sdk.__version__} is too old for dynamic workflows - "
    "re-run the install cell, then restart the kernel"
)
print(f"claude-agent-sdk {claude_agent_sdk.__version__}")

claude-agent-sdk 0.2.125


### OrbitCart Workspace 생성하기

가상의 회사는 온라인 cycling 장비 store인 **OrbitCart**입니다. 다음 cell은 disk에 작은 workspace를 작성합니다. 사실 주장 10개를 담은 투자자 업데이트 초안 `investor_update.md`와, 주장을 뒷받침해야 하는 ground truth(월별 매출 CSV, 고객 설문 결과, uptime 보고서, 언론 보도 파일)를 담은 `sources/` 디렉터리입니다.

Workflow가 실제로 잡아낼 대상을 갖도록 초안에 문제를 심어 두었습니다:

- **주장 1, 4, 5, 6은 출처와 모순됩니다.** 매출 CSV에서 Q2 매출은 $4.61M으로 Q1보다 28% 높지만, 초안은 $4.8M과 42%라고 주장합니다. 설문의 NPS는 71이 아닌 62입니다. Uptime 보고서의 분기 평균은 99.99%가 아닌 99.71%입니다(5월 6시간 incident 포함). TechPedal은 OrbitCart를 "가장 빠르게 성장하는 cycling 소매업체"가 아니라 "가장 빠르게 성장하는 온라인 cycling 소매업체 중 하나"라고 불렀습니다.
- **주장 7과 8은 검증할 수 없습니다.** 6월 flash sale이나 지원 해결 시간을 언급하는 출처가 없습니다.
- **주장 2, 3, 9, 10은 적어도 한 출처가 뒷받침합니다.**

Agent는 파일로 작업하므로 data는 disk에 있지만, agent가 정확히 무엇을 볼지 알 수 있도록 Notebook에서 생성합니다. 실행 후 workflow의 판단을 이 정답표와 비교하겠습니다.

In [3]:
WORKSPACE = Path("orbitcart_data").resolve()
SOURCES = WORKSPACE / "sources"
SOURCES.mkdir(parents=True, exist_ok=True)

# --- 투자자 업데이트 초안: fact-check workflow용 주장 10개 ---
(WORKSPACE / "investor_update.md").write_text("""\
# OrbitCart - Q2 2026 Investor Update (DRAFT)

*Prepared by the founding team. Not yet reviewed.*

Dear investors,

Q2 was OrbitCart's strongest quarter yet. Highlights:

1. **Revenue:** Q2 revenue reached **$4.8M, up 42% from Q1**.
2. **Order volume:** We processed **over 38,000 orders** in the quarter.
3. **Customers:** Our active customer base grew to **12,400**.
4. **Customer love:** We earned a **Net Promoter Score of 71**.
5. **Reliability:** The platform maintained **99.99% uptime** across the quarter.
6. **Press:** TechPedal called us **"the fastest-growing cycling retailer in North America."**
7. **Growth:** Our June flash sale was **the biggest single sales day in company history**.
8. **Support:** Customer support resolution time improved to **under 4 hours**.
9. **Catalog:** We launched **3 new product categories** this quarter.
10. **Trust:** **Zero customer data was lost** during the brief May service interruption.

We look forward to discussing these results at the board meeting.

- The OrbitCart team
""")

# --- Source 문서: 주장이 추적되어야 하는 ground truth ---
(SOURCES / "monthly_sales.csv").write_text("""\
month,revenue_usd,orders,new_customers
2026-01,1150000,9800,1020
2026-02,1180000,10100,990
2026-03,1270000,9900,1100
2026-04,1480000,12800,1410
2026-05,1520000,12400,1380
2026-06,1610000,13200,1640
""")

(SOURCES / "customer_survey.txt").write_text("""\
OrbitCart Customer Survey - Q2 2026 Results
Survey window: June 10-17, 2026 | Responses: 2,140

Net Promoter Score (NPS): 62  (Q1: 58)
Active customers at end of Q2: 12,400
Support satisfaction (CSAT): 4.4 / 5

Selected verbatim feedback:
- "Ordering was painless and the bike arrived two days early." (Pro customer, Denver)
- "Great selection, but I wish there were more filter options." (Free customer, Toronto)
- "The team plan made it easy to outfit our whole shop." (Team customer, Portland)
""")

(SOURCES / "uptime_report.txt").write_text("""\
OrbitCart Platform Reliability Report - Q2 2026

April 2026:  99.97% uptime, no incidents
May 2026:    99.18% uptime
  - May 14: checkout degraded for 5h 58m during a database failover.
    No customer data was lost. Root cause: connection pool exhaustion.
June 2026:   99.98% uptime, no incidents

Quarter average: 99.71%
""")

(SOURCES / "press_coverage.md").write_text("""\
# Press mentions - Q2 2026

## TechPedal, May 3, 2026
"OrbitCart has quietly become one of the fastest-growing online cycling retailers
in North America, on the strength of a no-friction checkout and a generous
returns window."

## Gear & Grease Weekly, June 21, 2026
"The Portland-based startup recently expanded into bike maintenance tools, indoor
training equipment, and cycling apparel - three categories that traditional bike
shops have struggled to move online."
""")

print(f"Workspace: {WORKSPACE.name}/")
print(f"Investor update claims: 10 | Source documents: {len(list(SOURCES.iterdir()))}")

Workspace: orbitcart_data/
Investor update claims: 10 | Source documents: 4


## Subagent와 Workflow 비교

Agent SDK는 이미 multi-step 작업을 실행하는 두 가지 방식을 제공하며, dynamic workflow가 세 번째를 추가합니다. 차이는 하나의 질문으로 귀결됩니다. **누가 계획을 보유하는가?**

|  | 단일 agent | Subagent | Dynamic workflow |
|---|---|---|---|
| **정의** | 하나의 agent가 단계별로 작업 | 주 agent가 시작하는 worker | Runtime이 실행하는 script |
| **다음 실행을 결정하는 주체** | Claude, turn마다 | Claude, turn마다 | Script |
| **중간 결과 위치** | Claude의 context window | 주 agent의 context window | Script 변수 |
| **반복 가능한 것** | Prompt | Worker 정의 | Orchestration 자체 |
| **규모** | Context window 하나 | Turn당 몇 개의 위임 작업 | 실행당 수십에서 수백 agent |
| **중단** | Turn 재시작 | Turn 재시작 | 같은 session에서 resume 가능 |

**Subagent**([Notebook 01](01_The_chief_of_staff_agent.ipynb)의 pattern)를 사용할 때 Claude가 orchestrator입니다. Turn마다 무엇을 위임할지 결정하며, 모든 부분을 위임하거나 마지막에 결과를 결합하거나 무언가를 검증한다는 보장이 없습니다. 모든 worker의 결과는 주 agent의 context로 돌아옵니다. 네 번의 위임에는 괜찮지만 400번에는 그렇지 않습니다. 계획도 Claude context에만 존재하므로 실행을 중단하면 처음부터 다시 작업합니다.

**Dynamic workflow**는 이 특성들을 각각 뒤집습니다. Script가 실행 대상을 결정하고, 중간 결과는 script 변수에 존재하며, script 자체는 읽고 저장하고 수정하고 재실행할 수 있는 artifact입니다. 많은 subagent를 시작하므로 대가는 token입니다. 작업이 **하나의 context window보다 크거나**, **구조로 검증을 강제**하고 싶거나, **orchestration 자체를 보존할 가치가 있을 때** 사용하세요.

## Dynamic Workflow의 구성

Agent SDK에서 workflow를 시작하려면 두 가지가 필요합니다:

1. `allowed_tools`에서 **`Workflow` tool 허용**. Claude가 workflow를 시작할 때 호출하는 tool이며, SDK는 이 목록의 tool을 자동 승인합니다.
2. **Prompt에서 workflow 요청**. "Workflow를 사용해서 …"처럼 평이한 표현으로 요청하세요. Claude는 이런 직접 요청을 turn별 작업 대신 script 기반 orchestration을 선택한 것으로 받아들입니다. (Interactive CLI에는 keyword trigger와 `/effort ultracode` mode가 추가되지만, SDK에서는 명확한 요청만 있으면 됩니다.)

Claude는 `Workflow` tool을 호출할 때 작업을 위해 작성한 완전한 JavaScript orchestration script를 전달합니다. Runtime은 script를 **background에서** 실행합니다. `query()` stream은 계속 열려 있고 agent 실행 중 진행 event가 돌아옵니다. 시작과 완료가 서로 다른 turn이므로 stream은 각 turn에 `ResultMessage`를 yield합니다. 마지막 것이 최종 결과를 담으며, 아래 `run_agent()` helper는 이를 유지합니다. SDK에는 interactive permission prompt가 없습니다. Tool 호출은 구성한 permission 규칙을 따르므로 명시적인 `allowed_tools` 목록을 설정합니다. (Allowlist 밖의 `Bash` 같은 읽기 전용 호출이 성공할 수도 있습니다. Non-interactive 실행은 읽기 전용 command를 자동 허용하고 나머지는 거부합니다.) Workflow가 시작한 subagent는 `acceptEdits` mode에서 실행되며 해당 allowlist를 상속합니다.

첫 실행 전에 알아둘 세 가지 사항:

- **Script는 실제 Artifact입니다.** Runtime은 `~/.claude/projects/`의 session 디렉터리 아래 파일에 작성하고 tool 결과에 경로를 포함하므로 아래 `run_agent()`가 가져올 수 있습니다. 실행 후 script를 읽습니다.
- **발견 사항은 보고서 파일이 아닌 Script로 반환됩니다.** Workflow agent는 발견한 내용을 반환값으로 전달하며, 이 방식으로 결과가 script 변수에 도달합니다. 실행 중 `SUMMARY.md` 같은 보고서형 파일을 쓰려 한 agent는 대신 콘텐츠를 반환하라는 지시를 받았습니다. 실제 작업 산출물(변환한 문서, 코드, data 파일)은 정상적으로 disk에 작성합니다.
- **실행은 실패하고 Resume할 수 있습니다.** 실행 도중 단계가 실패하면(예: agent가 structured-output 재시도 소진) log에 `[Failed]` 알림이 보이며 Claude는 보통 workflow를 다시 시작합니다. 이미 완료한 agent는 cached 결과를 반환하므로 재시도는 주로 실패한 단계를 다시 수행합니다. 그에 따라 비용이 증가할 수 있습니다. 이는 resume 기능이 작동하는 것이며 Notebook 오류가 아닙니다.
- **동작 및 한도.** Workflow는 최대 16개 agent를 동시에 실행하고(CPU core가 적은 컴퓨터에서는 더 적음), 실행당 최대 1,000개를 사용합니다. Orchestration script 자체는 filesystem이나 shell에 접근할 수 없고, 시작한 agent만 접근할 수 있습니다. 중지된 실행은 같은 session에서 resume할 수 있으며, 새 session에서는 새로 시작합니다.

이 폴더의 `agent_visualizer` helper가 다루지 않는 `TaskProgressMessage` 및 `TaskNotificationMessage` event를 workflow가 내보내므로, 여기서는 작은 logger를 직접 구현합니다. 다음 cell은 실행에 사용할 helper를 정의합니다. `run_agent()`는 query를 streaming하고 간결한 activity log를 출력하며, `workflow_options()`는 `ClaudeAgentOptions`를 구성하고, `show_script()`는 Claude가 생성한 orchestration script를 표시합니다.

In [4]:
# Workflow tool 결과는 영속적으로 저장한 script 경로와 task ID를 일반
# text로 언급하며, 이 regex가 추출합니다. Demo 편의 기능이며 안정적인 API는 아닙니다.
SCRIPT_PATH_RE = re.compile(r"Script file: (.+?\.js)")
TASK_ID_RE = re.compile(r"Task ID: (\S+)")


async def run_agent(prompt: str, options: ClaudeAgentOptions) -> dict:
    """Agent SDK로 query를 streaming하고 간결한 activity log를 출력합니다.

    최종 result text, 총비용, 그리고 workflow가 실행된 경우
    Claude가 생성한 orchestration script 경로를 담은 dict를 반환합니다.
    """
    info = {"result": None, "cost_usd": None, "script_path": None}
    last_tool_count = 0  # 진행 상황 줄 출력 빈도 제한
    started = time.monotonic()

    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    # Claude 설명의 첫 줄 표시
                    if narration := block.text.strip():
                        print(f"\nClaude: {narration.splitlines()[0]}")
                elif isinstance(block, ToolUseBlock):
                    if block.name == "Workflow":
                        script = block.input.get("script", "")
                        n_lines = len(script.splitlines())
                        print(
                            f"\n[Workflow tool] Claude wrote a {n_lines}-line orchestration script"
                        )
                    else:
                        print(f"  -> {block.name}")

        elif isinstance(msg, UserMessage):
            blocks = msg.content if isinstance(msg.content, list) else []
            for block in blocks:
                if not isinstance(block, ToolResultBlock):
                    continue
                text = block.content if isinstance(block.content, str) else str(block.content)
                if match := SCRIPT_PATH_RE.search(text):
                    info["script_path"] = match.group(1)
                    task = TASK_ID_RE.search(text)
                    task_id = task.group(1) if task else "?"
                    print(f"[Launched] Workflow running in the background (task {task_id})")

        elif isinstance(msg, TaskProgressMessage):
            # 진행 상황은 몇 초마다 도착하며, 새로운 agent tool 호출 약 8개마다 한 줄 출력
            tools = msg.usage.get("tool_uses", 0)
            if tools - last_tool_count >= 8:
                last_tool_count = tools
                tokens = msg.usage.get("total_tokens", 0)
                seconds = msg.usage.get("duration_ms", 0) / 1000
                print(
                    f"    ... {tools:>3} agent tool calls | {tokens:>9,} tokens | {seconds:>4.0f}s"
                )

        elif isinstance(msg, TaskNotificationMessage):
            print(f"[{msg.status.capitalize()}] {msg.summary}")

        elif isinstance(msg, ResultMessage):
            # Background workflow는 turn마다 결과 하나를 yield합니다. 시작 turn과
            # 완료 후 최종 turn입니다. 최신 결과를 유지하고 한 번 보고합니다.
            info["result"] = msg.result
            info["cost_usd"] = msg.total_cost_usd

    cost = f" | ${info['cost_usd']:.2f}" if info["cost_usd"] is not None else ""
    minutes = (time.monotonic() - started) / 60
    print(f"\n=== Run complete in {minutes:.1f} min{cost} ===")
    return info


def show_script(info: dict) -> None:
    """실행 중 Claude가 생성한 orchestration script를 rendering합니다."""
    path = info.get("script_path")
    if not path or not Path(path).exists():
        print("No workflow script was generated in this run.")
        return
    display(Markdown(f"```javascript\n{Path(path).read_text()}\n```"))


def workflow_options() -> ClaudeAgentOptions:
    """OrbitCart workspace에서 작업하는 workflow 활성화 agent의 ClaudeAgentOptions."""
    return ClaudeAgentOptions(
        cwd=str(WORKSPACE),
        model=MODEL,
        allowed_tools=["Read", "Write", "Edit", "Glob", "Grep", "Workflow"],
        permission_mode="acceptEdits",
        max_turns=40,
    )

## 예제: 투자자 업데이트 Fact-check하기

**작업:** 모든 주장을 source 문서로 추적하기 전에는 법무팀이 `investor_update.md`를 승인하지 않습니다. 주장 10개와 source 파일 네 개가 있습니다.

**Workflow를 사용하는 이유:** 하나의 agent가 주장 10개를 확인하면 모든 source 문서가 context에 쌓이고, 7번 주장쯤부터 대충 훑기 시작합니다. "발견 사항을 다시 확인하세요"도 지침일 뿐이므로 context 압박을 받으면 생략됩니다. 아래 workflow는 검증을 *구조적인 것*으로 만듭니다:

1. **추출:** Agent 하나가 초안을 읽고 모든 사실 주장을 추출
2. **검증:** *주장마다* agent 하나가 각자의 깨끗한 context에서 출처를 새로 읽음
3. **Adversarial 검사:** Skeptic agent가 수락 전에 모든 판단을 반박
4. **보고:** Agent 하나가 판단을 go/no-go 보고서로 취합

이것이 **fan-out + adversarial verification** pattern입니다. 아래 prompt의 많은 부분이 작업 자체보다 작업의 *형태*를 설명한다는 점에 주목하세요. Workflow에서 prompt는 원하는 harness를 설명합니다.

In [5]:
FACT_CHECK_PROMPT = """\
Use a workflow to fact-check the draft investor update at investor_update.md against
the source documents in sources/.

Structure the workflow exactly like this:

1. EXTRACT: one agent reads investor_update.md and extracts its ten numbered
   highlights as structured output (claim number, claim text). Keep the draft's own
   numbering 1-10 and treat each numbered highlight as exactly one claim, even when it
   bundles two figures.

2. VERIFY: one agent per claim, running in parallel. Each verifier reads the source
   documents in sources/ and returns a verdict as structured output:
   - "confirmed" if a source directly supports the claim (quote the supporting line)
   - "contradicted" if a source conflicts with it (quote the conflicting line and
     state the correct figure)
   - "unverifiable" if no source covers it
   Verifiers must quote the exact lines they relied on. Pay attention to subtle
   differences between what a source says and what the draft claims it says.

3. SKEPTIC: for every "confirmed" verdict, one skeptic agent re-reads the cited
   source and tries to refute the confirmation. If the skeptic finds the citation
   does not actually support the claim, the verdict changes to "contradicted".

4. REPORT: one final agent compiles a markdown fact-check report: a table of every
   claim with its verdict and evidence, then a summary of what must be fixed before
   the update can be sent. Return this report as the workflow result.

The working directory is already set to the folder containing these files. Refer to
every file by relative path (e.g. investor_update.md, sources/monthly_sales.csv) in
the script and in agent prompts; do not embed absolute paths.
"""

factcheck_info = await run_agent(FACT_CHECK_PROMPT, workflow_options())

  -> Bash


  -> Read


  -> Read


  -> Read


  -> Read


...

    ...  48 agent tool calls |   543,126 tokens |   30s


    ...  56 agent tool calls |   549,412 tokens |   42s


[Completed] Dynamic workflow "Fact-check the OrbitCart investor update against source documents" completed



Claude: The fact-check workflow finished. Here's the report:



=== Run complete in 2.5 min | $3.29 ===


In [6]:
display(Markdown(factcheck_info["result"] or "_(no result returned)_"))

The fact-check workflow finished. Here's the report:

# OrbitCart Q2 2026 Investor Update — Fact-Check Report

## Claim-by-Claim Verification

| # | Claim | Verdict | Evidence |
|---|---|---|---|
| 1 | Revenue $4.8M, up 42% from Q1 | ❌ Contradicted | `monthly_sales.csv`: Q2 = $1.48M+$1.52M+$1.61M = **$4.61M**; Q1 = $3.6M → growth ≈ **28%**, not 42% |
| 2 | Over 38,000 orders | ✅ Confirmed | `monthly_sales.csv`: 12,800+12,400+13,200 = **38,400 orders** |
| 3 | Active customers 12,400 | ✅ Confirmed | `customer_survey.txt`: "Active customers at end of Q2: 12,400" |
| 4 | NPS of 71 | ❌ Contradicted | `customer_survey.txt`: "Net Promoter Score (NPS): **62** (Q1: 58)" |
| 5 | 99.99% uptime | ❌ Contradicted | `uptime_report.txt`: "Quarter average: **99.71%**" (May was only 99.18%, with a 5h58m outage) |
| 6 | TechPedal: "the fastest-growing cycling retailer in North America" | ❌ Contradicted | `press_coverage.md` actual quote: "**one of** the fastest-growing **online** cycling retailers in North America" — misquoted, overstated |
| 7 | June flash sale = biggest sales day in company history | ⚠️ Unverifiable | No source has daily/flash-sale data |
| 8 | Support resolution under 4 hours | ⚠️ Unverifiable | No source mentions resolution time |
| 9 | 3 new product categories launched | ✅ Confirmed | `press_coverage.md`: bike maintenance tools, indoor training equipment, cycling apparel |
| 10 | "Brief" May interruption, zero data lost | ❌ Contradicted (framing) | `uptime_report.txt`: outage lasted **5h 58m** — not "brief"; the "zero data lost" part is accurate |

## Summary — must fix before sending

- **#1 Revenue:** correct to **$4.61M, +28% QoQ** (not $4.8M/42%)
- **#4 NPS:** correct to **62** (not 71)
- **#5 Uptime:** correct to **99.71% average** (not 99.99%); consider disclosing the May 14 outage
- **#6 Press quote:** fix misquote — TechPedal said "one of the fastest-growing **online** cycling retailers," not "the fastest-growing cycling retailer"
- **#10 Framing:** the outage was **~6 hours**, not "brief" — reword (data-loss claim itself is accurate)
- **#7 Flash sale:** no supporting source — provide data or remove/soften
- **#8 Support time:** no supporting source — provide data or remove/soften

**Safe as-is:** #2 (38,400 orders), #3 (12,400 customers), #9 (3 new categories).

Data에 심어 둔 문제를 반영한 정답표와 판단을 비교하세요:

| 주장 | 심어 둔 사실 |
|---|---|
| 1. "$4.8M, 42% 증가" | **모순됨.** CSV의 Q2 합계는 $4.61M이며 Q1의 $3.6M보다 28% 증가 |
| 4. "NPS 71" | **모순됨.** 설문은 62라고 함 |
| 5. "99.99% uptime" | **모순됨.** Reliability 보고서는 99.71%이며 5월에 6시간 incident 발생 |
| 6. TechPedal 인용 | **모순됨.** 기사는 "가장 빠르게 성장하는 *온라인* cycling 소매업체 *중 하나*"라고 하며 "가장 빠르게 성장하는"이라고 하지 않음 |
| 7. Flash sale, 8. 지원 시간 | **검증 불가.** 어느 출처도 이를 다루지 않음 |
| 2, 3, 9, 10 | **출처가 뒷받침함** |

6번 주장은 자세히 살펴볼 가치가 있습니다. 서두르는 단일 agent는 "가장 빠르게 성장하는"과 "가장 빠르게 성장하는 업체 중 하나"의 차이를 놓치는 경우가 많습니다. Context에 다른 내용이 없는 전담 verifier가 skeptic의 반박까지 받으면 속아 넘어가기 훨씬 어렵습니다. 이 정확성은 더 똑똑한 model이 아닌 workflow 구조에서 나옵니다.

뒷받침되는 주장에 대해 workflow가 정답표보다 *엄격한* 판단을 내리더라도 놀라지 마세요. 주장 3, 9, 10은 skeptic pass가 반박하는 종류입니다. 설문은 12,400 수치를 확인하지만 "증가했다"에 필요한 이전 baseline은 어느 출처도 확립하지 않습니다(주장 3). 언론 기사는 회사가 세 category로 "최근 확장했다"고 하지만 출시가 *Q2 내에* 발생했는지 확인하지 않습니다(주장 9). 5월에 data가 유실되지는 않았지만 skeptic은 6시간 outage를 "짧은" 중단이라고 부르는 데 이의를 제기할 수 있습니다(주장 10). 실행에서 확인하든 문제로 표시하든 adversarial 검사가 제 역할을 하는 것입니다. 지나치게 엄격한 표시는 사람의 검토 1분을 소모하지만, 지나치게 느슨한 통과는 투자자의 신뢰를 소모합니다.

### Claude가 작성한 Script 읽기

가장 많은 것을 배울 수 있는 artifact는 작업 구성 방식을 보여 주는 script 자체입니다. `Workflow` tool 결과에는 영속적으로 저장한 script의 경로가 포함되었으며 `run_agent()`가 이를 가져왔습니다. 읽으면서 verifier가 structured 판단을 반환하게 하는 `schema` 옵션과 phase 사이의 일반 JavaScript 로직에 주목하세요. 확인된 판단만 skeptic으로 보내는 데는 token이 전혀 들지 않습니다.

In [7]:
show_script(factcheck_info)

```javascript
export const meta = {
  name: 'orbitcart-fact-check',
  description: 'Fact-check the OrbitCart investor update against source documents',
  phases: [
    { title: 'Extract' },
    { title: 'Verify' },
    { title: 'Skeptic' },
    { title: 'Report' },
  ],
}

const EXTRACT_SCHEMA = {
  type: 'object',
  properties: {
    claims: {
      type: 'array',
      items: {
        type: 'object',
        properties: {
          number: { type: 'integer' },
          text: { type: 'string' },
        },
        required: ['number', 'text'],
      },
    },
  },
  required: ['claims'],
}

const VERIFY_SCHEMA = {
  type: 'object',
  properties: {
    number: { type: 'integer' },
    text: { type: 'string' },
    verdict: { type: 'string', enum: ['confirmed', 'contradicted', 'unverifiable'] },
    source_file: { type: 'string', description: 'relative path of the source file relied on, or empty string' },
    quote: { type: 'string', description: 'exact quoted line(s) from the source relied on; empty string if unverifiable' },
    correct_figure: { type: 'string', description: 'the correct figure/fact per the source, only if contradicted; else empty string' },
    reasoning: { type: 'string' },
  },
  required: ['number', 'text', 'verdict', 'source_file', 'quote', 'correct_figure', 'reasoning'],
}

const SKEPTIC_SCHEMA = {
  type: 'object',
  properties: {
    number: { type: 'integer' },
    final_verdict: { type: 'string', enum: ['confirmed', 'contradicted'] },
    source_file: { type: 'string' },
    quote: { type: 'string' },
    correct_figure: { type: 'string', description: 'only if downgraded to contradicted; else empty string' },
    reasoning: { type: 'string' },
  },
  required: ['number', 'final_verdict', 'source_file', 'quote', 'correct_figure', 'reasoning'],
}

// 1. EXTRACT
phase('Extract')
const extraction = await agent(
  `Read the file investor_update.md in the current working directory. It is a draft investor update ` +
  `containing a numbered list of ten highlights (1-10). Extract each numbered highlight as exactly one ` +
  `claim, preserving the draft's own numbering 1-10. Treat each numbered item as exactly one claim even ` +
  `when it bundles two figures together (e.g. an item that states both a dollar amount and a percentage ` +
  `is still ONE claim, not two). Return the claim text verbatim as written in the draft (including bolded ` +
  `figures), for all ten items.`,
  { label: 'extract', schema: EXTRACT_SCHEMA }
)

const claims = extraction.claims.sort((a, b) => a.number - b.number)
log(`Extracted ${claims.length} claims from investor_update.md`)

// 2. VERIFY (parallel, one per claim) then 3. SKEPTIC (only for confirmed verdicts)
const results = await pipeline(
  claims,
  (claim) => agent(
    `You are fact-checking one highlight from OrbitCart's draft Q2 2026 investor update against source ` +
    `documents. The source documents live in the sources/ directory of the current working directory: ` +
    `sources/monthly_sales.csv, sources/customer_survey.txt, sources/press_coverage.md, and ` +
    `sources/uptime_report.txt. Read all four files.\n\n` +
    `Claim #${claim.number}: "${claim.text}"\n\n` +
    `Determine a verdict:\n` +
    `- "confirmed" if a source directly supports the claim as written. Quote the exact supporting line(s).\n` +
    `- "contradicted" if a source conflicts with the claim (a different number, a different attribution, ` +
    `a stat that doesn't match, etc). Quote the exact conflicting line(s) and state the correct figure/fact ` +
    `per the source.\n` +
    `- "unverifiable" if no source covers this claim at all.\n\n` +
    `Be precise and skeptical: pay close attention to subtle differences between what a source actually ` +
    `says and what the draft claims it says (e.g. a claim may bundle a true figure with a false one, cite ` +
    `the wrong quarter, misattribute a quote, round in a misleading way, or overstate what a source covers). ` +
    `If the claim bundles two figures and only one is supported, that counts as "contradicted", not ` +
    `"confirmed" — quote the source and explain exactly which part is wrong. Quote exact lines from the ` +
    `source file(s), do not paraphrase the quote field.`,
    { label: `verify-${claim.number}`, phase: 'Verify', schema: VERIFY_SCHEMA }
  ),
  (verifyResult, claim) => {
    if (!verifyResult || verifyResult.verdict !== 'confirmed') {
      return Promise.resolve(verifyResult ? { ...verifyResult, skeptic_reviewed: false } : null)
    }
    return agent(
      `You are a skeptical fact-checking reviewer. Another verifier examined claim #${claim.number} from ` +
      `OrbitCart's draft Q2 2026 investor update and marked it "confirmed", citing this evidence:\n\n` +
      `Claim: "${claim.text}"\n` +
      `Cited source file: ${verifyResult.source_file}\n` +
      `Cited quote: "${verifyResult.quote}"\n` +
      `Verifier reasoning: ${verifyResult.reasoning}\n\n` +
      `Re-read the cited source file (path relative to the current working directory, inside sources/) ` +
      `yourself and try hard to REFUTE this confirmation. Look for: the quote being taken out of context, ` +
      `the quote actually supporting a different number/fact than the claim states, the claim bundling a ` +
      `second figure that the quote does NOT cover, misattribution, wrong time period, or rounding that ` +
      `changes the meaning. If you find the citation does NOT actually fully support the claim, set ` +
      `final_verdict to "contradicted", quote the real conflicting/limiting line, and state the correct ` +
      `figure or fact. Only keep final_verdict "confirmed" if the citation genuinely and fully supports the ` +
      `claim as written after your adversarial check.`,
      { label: `skeptic-${claim.number}`, phase: 'Skeptic', schema: SKEPTIC_SCHEMA }
    ).then((skepticResult) => {
      if (!skepticResult) return { ...verifyResult, skeptic_reviewed: false }
      if (skepticResult.final_verdict === 'contradicted') {
        return {
          ...verifyResult,
          verdict: 'contradicted',
          source_file: skepticResult.source_file || verifyResult.source_file,
          quote: skepticResult.quote || verifyResult.quote,
          correct_figure: skepticResult.correct_figure || verifyResult.correct_figure,
          reasoning: `Skeptic overturned confirmation: ${skepticResult.reasoning}`,
          skeptic_reviewed: true,
        }
      }
      return { ...verifyResult, skeptic_reviewed: true, reasoning: `${verifyResult.reasoning} (skeptic upheld: ${skepticResult.reasoning})` }
    })
  }
)

const finalResults = results.filter(Boolean).sort((a, b) => a.number - b.number)
log(`Verification complete for ${finalResults.length} claims`)

// 4. REPORT
phase('Report')
const report = await agent(
  `Compile a markdown fact-check report for OrbitCart's draft Q2 2026 investor update ` +
  `(investor_update.md), based on this verification data (JSON, one entry per numbered claim, already ` +
  `including any skeptic-review overturns):\n\n${JSON.stringify(finalResults, null, 2)}\n\n` +
  `Produce a markdown report with:\n` +
  `1. A table with columns: Claim #, Claim, Verdict, Evidence (the source file and quote, or "no source ` +
  `covers this" for unverifiable claims, or the conflicting quote + correct figure for contradicted claims).\n` +
  `2. A "Summary" section listing exactly what must be fixed before the update can be sent — call out every ` +
  `"contradicted" claim with the correct figure/fact to use instead, and every "unverifiable" claim that ` +
  `needs a source or should be softened/removed. If everything is confirmed, say so plainly.\n\n` +
  `Return only the finished markdown report as your final answer.`,
  { label: 'report' }
)

return report

```

### Workflow Script 읽는 방법

모든 workflow script는 같은 구성 요소를 사용합니다. 전체 옵션은 [Agent SDK reference](https://code.claude.com/docs/en/agent-sdk/typescript)의 Workflow tool 항목에 문서화되어 있지만, 위에서 본 내용에 일반적인 primitive를 대응하면 대부분 배울 수 있습니다:

| Primitive | 기능 |
|---|---|
| `export const meta = {name, description, phases}` | Workflow 이름과 phase 선언. 진행 UI는 agent를 phase별로 묶음 |
| `agent(prompt, options)` | **깨끗한 context**를 가진 subagent 하나를 시작. 주어진 prompt 문자열만 보고 다른 것은 보지 않음. Options에서 `label`, `phase`, structured output용 JSON `schema`, `model` 설정 가능 |
| `parallel([...])` | Agent batch를 동시에 실행하고 **모두 완료될 때까지 대기**(barrier)한 뒤 진행 |
| `pipeline(items, stage1, stage2, ...)` | 각 항목을 단계 순서대로 **독립적으로** 실행하여, 항목 B가 1단계에 있는 동안 항목 A는 2단계에 있을 수 있음 |
| `phase("...")` | 이후 agent가 속하는 phase 표시 |
| 호출 사이의 일반 JavaScript | Filtering, 중복 제거, 병합, loop를 모두 정확하고 즉각적으로 수행하며 token 비용 없음 |
| `return {...}` | Script가 반환하는 내용이 session으로 돌아옴 |

이 설계에서 세 가지 특성이 도출됩니다:

1. **계획은 코드로 강제됩니다.** Script의 control flow가 그렇게 순서를 정하므로 skeptic pass가 verifier 이후 실행됩니다. Claude가 지침을 기억하는지에 의존하지 않습니다.
2. **Claude의 Context는 깨끗하게 유지됩니다.** 모든 판단 및 인용한 source 행은 script 변수에 있었습니다. 최종 취합 보고서만 workflow를 시작한 session에 도달했습니다.
3. **Script는 자신의 것입니다.** Disk의 파일입니다. 읽고(방금 읽음), 수정하고, 프로젝트의 `.claude/workflows/` 디렉터리에 저장하며, 이름으로 재실행할 수 있습니다. Orchestration이 팀의 재사용 가능한 자산이 됩니다.

## Workflow를 사용할 때

Dynamic workflow는 난이도 곡선의 최상단에 있으며, 그 아래에서는 단일 agent와 subagent가 여전히 올바른 tool입니다. 합리적인 선택 기준은 다음과 같습니다:

| 선택할 것... | 경우... |
|---|---|
| **단일 agent** | 작업이 하나의 context window에 맞고 독립적인 검증이 필요하지 않음. 대부분의 작업이 여기에 속함. |
| **Subagent** | 몇 명의 전문 worker(researcher, reviewer)가 필요하며 결과가 들어올 때 주 agent가 계획을 조정해야 함 |
| **Dynamic workflow** | 작업이 하나의 context window보다 큼(많은 항목, 많은 출처), 생략할 수 없는 검증이 필요함, 또는 반복할 process임 |

이 Notebook에서 기억할 세 가지 비용 제어 습관:

1. **Prompt에 구조를 명시하세요.** 설명한 harness가 제공되는 harness이며, 검증량과 저렴한 model을 쓸 수 있는 단계도 포함됩니다.
2. **실행별 비용을 확인하세요.** 위 실행에서도 출력했습니다. 모든 것을 검증하는 workflow는 아무것도 검증하지 않는 workflow보다 비싸며, 이 조절은 prompt에서 설정합니다.
3. **Data 규모를 현실적으로 유지하세요.** 이 Notebook이 몇 분 안에 실행되도록 주장 10개와 출처 네 개를 사용했습니다. 같은 구조는 수백 항목으로 확장됩니다. Script는 크기를 신경 쓰지 않지만 token 예산은 그렇지 않습니다.

## 정리

이 Notebook이 생성한 workspace를 삭제합니다. 파일과 fact-check 증거를 먼저 살펴보고 싶다면 이 cell을 건너뛰고, fact-check를 다시 실행하기 전에 workspace 생성 cell을 재실행하세요.

In [8]:
shutil.rmtree(WORKSPACE, ignore_errors=True)
print("Workspace removed.")

Workspace removed.


## 배운 내용

소개에서 제시한 목표에 대응하면:

- **Workflow 사용 시점:** 작업이 하나의 context window보다 크거나, 검증이 선택 사항 대신 구조적이어야 하거나, orchestration을 보존할 가치가 있을 때. 그 아래에서는 단일 agent 및 subagent가 여전히 올바른 tool입니다.
- **Agent SDK에서 시작하는 방법:** `allowed_tools`에 `Workflow` tool을 허용하고, prompt에서 workflow를 요청하며, 실행(시작, 진행 event, 완료)을 `query()`로 streaming합니다.
- **Claude가 작성한 Script 읽는 방법:** `meta`는 phase를 선언하고, `agent()`는 깨끗한 context의 worker를 시작하고, `parallel()`은 barrier이며, `pipeline()`은 항목을 단계에 따라 streaming하고, 일반 JavaScript가 사이의 로직을 무료로 처리합니다. Fact-check에는 **fan-out + adversarial verification**을 사용했으며, 같은 primitive로 pipeline, tournament, triage queue를 구성할 수 있습니다.
- **Workflow를 재사용 가능하게 만드는 방법:** Script는 파일입니다. 프로젝트의 `.claude/workflows/` 디렉터리에 저장하고 commit하면, 해당 프로젝트에서 작업하는 누구나(또는 어떤 agent든) 같은 orchestration을 이름으로 재실행할 수 있습니다.

## 다음 단계

- **자신의 Data에 Pattern을 적용하세요.** Fact-check workflow는 보고서, marketing 문구, 문서처럼 주장과 출처를 대조하는 모든 문제에 작동합니다. 같은 구조(추출, 병렬 검증, 반박, 취합)는 콘텐츠 migration, triage queue, "판단에 따라 많은 항목 순위화" 문제에도 적용됩니다.
- **Claude Code CLI에서 Workflow를 시험하세요.** `/workflows`로 실행을 interactive하게 확인하고 keystroke로 저장할 수 있습니다. [Dynamic workflow 문서](https://code.claude.com/docs/en/workflows)를 참고하세요.
- **이 Series의 다른 Orchestration 방식과 비교하세요:** [Notebook 01](01_The_chief_of_staff_agent.ipynb)은 subagent를 상세히 다루며, [Building Effective Agents pattern](../patterns/agents/README.md)은 같은 orchestration 아이디어를 Messages API로 직접 구현한 예를 보여 줍니다.
- **Anthropic의 Multi-agent System Engineering Blog**, [How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system)을 읽고 multi-agent 구성이 언제 단일 agent보다 나은지와 비용에 대한 연구를 살펴보세요.